Historical study notebook. Its original input is unavailable in this public edition, so this is not a runnable public demo. It may require an input with the original structure and dependencies. This work used supplied learning resources and starter material; retained attribution appears with the notebook content.



In [ ]:
!pip install --upgrade -q gradio PyPDF2

In [ ]:
import gradio as gr
from PyPDF2 import PdfReader

# --- Mock backend (Step 2 will replace these with the real RAG pipeline) ---
def process_pdf(file):
    if file is None:
        return "⚠️ Please upload a PDF first."
    reader = PdfReader(file.name)
    n_pages = len(reader.pages)
    return f"✅ Processed '{file.name.split('/')[-1]}' — {n_pages} pages indexed and ready for questions."

def handle_chat(message, history, mode):
    if mode == "Summarize":
        reply = f"[Summary mode] (mock) A short summary related to: '{message}'"
    elif mode == "Extract":
        reply = f"[Extract mode] (mock) Extracted fields relevant to: '{message}'"
    else:  # Q&A
        reply = f"[Q&A mode] (mock) This is a placeholder answer to: '{message}'"
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply},
    ]
    return history, ""


In [ ]:
with gr.Blocks(title="Pharma Document Q&A Chatbot") as demo:
    gr.Markdown("## 📄 Pharma Document Q&A Chatbot\nUpload a pharmaceutical PDF and ask questions about it.")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
            mode = gr.Dropdown(
                ["Q&A", "Summarize", "Extract"],
                value="Q&A", label="Mode"
            )
            process_btn = gr.Button("🔄 Process Document", variant="primary")
            status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", height=450)   # no type= in Gradio 6.0
            user_input = gr.Textbox(placeholder="Ask a question about your document...", label="Your Question")
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat")

    process_btn.click(process_pdf, inputs=pdf_input, outputs=status)
    send_btn.click(handle_chat, inputs=[user_input, chatbot, mode], outputs=[chatbot, user_input])
    user_input.submit(handle_chat, inputs=[user_input, chatbot, mode], outputs=[chatbot, user_input])
    clear_btn.click(lambda: [], outputs=chatbot)

demo.launch(share=True, theme=gr.themes.Soft())   # theme moved here in Gradio 6.0